# derived_8.4-formal-eval-1.0 — Formal Statistical Evaluation of the Two-Regime Clustering Model

## Objective

Publication-oriented statistical evaluation of the claim established in `derived_8.4-eval-1.1` / `-1.3`:
**a two-regime (KMeans k=2) clustering model beats the single-regime global model and the trained-gating model**,
on the frozen temporal split (2023–2025 test) and under leave-one-station-out (LOSO) spatial generalization.

All tables and figures in this notebook are generated solely from the executed experiment artifacts
(`temporal_seed_*.csv`, `loso_seed_*.csv`, per-seed predictions, `pinned_configurations.json`,
`val_selected_deltas.json`). The README tables are copied verbatim from this notebook's stdout.

# Protocol summary

- **Temporal (primary):** experts trained on trainval (train 2017–2020 + val 2021–2022, 14,608 rows),
  evaluated on the frozen test set (2023–2025, 6,620 rows, 7 WA stations), **30 random seeds**.
  Seed 42 is included as an exact replication anchor against eval-1.1 / eval-1.3.
- **LOSO (secondary):** same 20 configurations × **5 seeds** × 7 held-out stations; the router is refitted
  per fold on the 6-station trainval (no held-out-station leakage into routing).
- **Delta-robustness:** per-regime delta features from three selection sources — *test-selected* (eval-1.1
  protocol, pinned), *val-selected* (re-ranked on validation-period residuals, `select_deltas_val.py`),
  *none* (c0 = c1 = 0).
- **Seed scope:** only the XGBoost expert regressors' `random_state` varies; routers (KMeans / gating
  classifier) stay at seed 42 because the delta additions are tied to the seed-42 cluster labels
  (a KMeans label flip across seeds would apply cluster-1 additions to the wrong regime).
- **Statistics:** seed-level (mean ± std, median, 95% t-CI, paired t-test, Wilcoxon signed-rank,
  % seeds A better), sample-level (paired cluster bootstrap over (station, month) blocks, percentile
  95% CI + bootstrap p), Benjamini–Hochberg FDR over the reported comparison family, and LOSO
  per-station win counts + two-sided sign test (n = 7 — low power; 6/7 wins is not significant at 0.05).

**Known leakage caveats (paper):** the 54-feature backbone and V0-50 features were selected targeting the
test period (`derived_8.4-feature-selection-2.0`); not re-fixable under the frozen-split constraint —
shared by all compared models, so relative conclusions are less affected; the delta ablation bounds the
residual impact. XGBoost hyperparameters come from earlier test-era tuning (shared across all models).
2025 test coverage is partial at several stations.

# Setup

Loads the experiment config, the derived_8.4 split, the pinned configurations, and the raw per-seed
metric CSVs written by `run_temporal.py` / `run_loso.py`.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# cwd-robust: nb execute may run the kernel from the notebook's own directory.
if (Path.cwd() / "config.yaml").exists():
    EXP_DIR = Path.cwd()
    PROJECT_ROOT = EXP_DIR.parents[2]
else:
    EXP_DIR = Path.cwd() / "experiment" / "derived_8.4-formal-eval-1.0"
    PROJECT_ROOT = Path.cwd().parent
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

from eval_formal.configs import config_frame
from eval_formal.data import load_experiment_data
from eval_formal import stats as st
from eval_formal import plots

config = yaml.safe_load((EXP_DIR / "config.yaml").read_text())
data = load_experiment_data(PROJECT_ROOT, config)
cfg = pd.read_csv(EXP_DIR / "pinned_configs.csv")

temporal = pd.read_csv(EXP_DIR / "temporal_seed_summary.csv")
temporal_station = pd.read_csv(EXP_DIR / "temporal_seed_station.csv")
temporal_year = pd.read_csv(EXP_DIR / "temporal_seed_year.csv")
loso = pd.read_csv(EXP_DIR / "loso_seed_station.csv")
loso_year = pd.read_csv(EXP_DIR / "loso_seed_year.csv")

print(f"[Data] TrainVal={len(data.trainval)} Test={len(data.test)} "
      f"({len(data.test['station_id'].unique())} stations in test)")
print(f"[Configs] {len(cfg)} pinned configurations "
      f"({cfg['delta_source'].value_counts().to_dict()})")
print(f"[Temporal seeds] {temporal['seed'].nunique()} done, "
      f"expected {len(config['seeds']['temporal'])}")
print(f"[LOSO seeds] {loso['seed'].nunique()} done, expected {len(config['seeds']['loso'])}")
n_complete = temporal.groupby("config_id")["seed"].nunique()
print("[Temporal completion per config]")
print(n_complete.to_string())


[Data] TrainVal=14608 Test=6620 (7 stations in test)
[Configs] 4 pinned configurations ({'test': 2, 'global': 1, 'val': 1})
[Temporal seeds] 2 done, expected 30
[LOSO seeds] 2 done, expected 5
[Temporal completion per config]
config_id
Clustering_V0_Full_k2_c0_0_c1_10    2
Clustering_V0_Full_k2_val_winner    2
Global_Single_54                    2
Trained_Gating_k2_c0_5_c1_10        2


# Temporal results — per-configuration seed-level summary

For every configuration and metric (R², RMSE, MAE, bias): mean ± std and median over the 30 seeds,
95% t-confidence interval of the mean (df = 29), min/max. The seed variation quantifies **fitting
stochasticity** under the frozen split (subsample/colsample sampling); test-set sampling variability
is quantified separately by the cluster bootstrap below.

In [2]:
METRICS = ["r2", "rmse", "mae", "bias"]
LOWER_BETTER = {"rmse", "mae", "bias", "ubrmse"}

rows = []
for cid in cfg["config_id"]:
    sub = temporal[temporal["config_id"] == cid]
    for m in METRICS:
        s = st.seed_summary(sub[m])
        rows.append({
            "config_id": cid,
            "metric": m.upper(),
            "n_seeds": int(s["n"]),
            "mean": s["mean"],
            "std": s["std"],
            "median": s["median"],
            "min": s["min"],
            "max": s["max"],
            "ci_low": s["ci_low"],
            "ci_high": s["ci_high"],
        })
summary = pd.DataFrame(rows)
summary = summary.merge(cfg[["config_id", "config_label", "strategy_name", "delta_source"]],
                        on="config_id", how="left")
summary.to_csv(EXP_DIR / "temporal_config_summary.csv", index=False)
print(f"[Artifacts] Wrote temporal_config_summary.csv ({len(summary)} rows)")
print()
print("R2 (mean ± std over seeds, [95% t-CI]) — sorted by mean")
r2tab = summary[summary["metric"] == "R2"].sort_values("mean", ascending=False).copy()
r2tab["mean_std"] = r2tab.apply(
    lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
r2tab["ci"] = r2tab.apply(
    lambda r: f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]", axis=1)
print(r2tab[["config_label", "delta_source", "n_seeds", "mean_std", "median", "ci"]]
      .to_markdown(index=False))

[Artifacts] Wrote temporal_config_summary.csv (16 rows)

R2 (mean ± std over seeds, [95% t-CI]) — sorted by mean
| config_label                       | delta_source   |   n_seeds | mean_std        |   median | ci               |
|:-----------------------------------|:---------------|----------:|:----------------|---------:|:-----------------|
| Trained_Gating_k2  c0=5, c1=10     | test           |         2 | 0.6574 ± 0.0023 | 0.65736  | [0.6368, 0.6779] |
| Clustering_V0_Full_k2  c0=0, c1=10 | test           |         2 | 0.5084 ± 0.0060 | 0.5084   | [0.4542, 0.5626] |
| Global_Single_54                   | global         |         2 | 0.4918 ± 0.0004 | 0.491799 | [0.4879, 0.4957] |
| Clustering_V0_Full_k2  c0=5, c1=5  | val            |         2 | 0.4666 ± 0.0017 | 0.466564 | [0.4516, 0.4816] |


# Temporal results — RMSE / MAE / bias seed-level summary

Same summary for the remaining reported metrics. Lower is better for all three; units are m³/m³
(volumetric soil moisture at 5 cm).

In [3]:
for metric in ["RMSE", "MAE", "BIAS"]:
    tab = summary[summary["metric"] == metric].sort_values("mean", ascending=True).copy()
    tab["mean_std"] = tab.apply(lambda r: f"{r['mean']:.5f} ± {r['std']:.5f}", axis=1)
    tab["ci"] = tab.apply(lambda r: f"[{r['ci_low']:.5f}, {r['ci_high']:.5f}]", axis=1)
    print(f"\n{metric} (mean ± std over seeds, [95% t-CI]) — sorted by mean (lower is better)")
    print(tab[["config_label", "delta_source", "n_seeds", "mean_std", "median", "ci"]]
          .to_markdown(index=False))


RMSE (mean ± std over seeds, [95% t-CI]) — sorted by mean (lower is better)
| config_label                       | delta_source   |   n_seeds | mean_std          |    median | ci                 |
|:-----------------------------------|:---------------|----------:|:------------------|----------:|:-------------------|
| Trained_Gating_k2  c0=5, c1=10     | test           |         2 | 0.05963 ± 0.00020 | 0.0596285 | [0.05784, 0.06141] |
| Clustering_V0_Full_k2  c0=0, c1=10 | test           |         2 | 0.07142 ± 0.00044 | 0.071423  | [0.06749, 0.07536] |
| Global_Single_54                   | global         |         2 | 0.07262 ± 0.00003 | 0.0726195 | [0.07234, 0.07290] |
| Clustering_V0_Full_k2  c0=5, c1=5  | val            |         2 | 0.07440 ± 0.00012 | 0.0744006 | [0.07335, 0.07545] |

MAE (mean ± std over seeds, [95% t-CI]) — sorted by mean (lower is better)
| config_label                       | delta_source   |   n_seeds | mean_std          |    median | ci                 |


# Temporal pairwise tests — focused comparison family

Pre-specified family (per metric): each model vs {Global_Single_54, Baseline_V0_50,
Trained_Gating_k2_c0_5_c1_10}, plus within-strategy delta ablations (test-selected vs val-selected vs
none). Reported per pair: mean difference (A − B, seed-level), 95% t-CI, paired t-test p,
Wilcoxon signed-rank p, % seeds where A is better (direction-aware), and the Benjamini–Hochberg
q-value over the family. The full all-pairwise seed-level matrix is exported separately.

In [4]:
def cmp_rows(seed_df, a, b, metric):
    pa = seed_df[seed_df["config_id"] == a].set_index("seed")[metric].sort_index()
    pb = seed_df[seed_df["config_id"] == b].set_index("seed")[metric].sort_index()
    common = pa.index.intersection(pb.index)
    if len(common) < 2:
        return None
    x = pa.loc[common]; y = pb.loc[common]
    better = lambda d: (d > 0) if metric not in LOWER_BETTER else (d < 0)
    pt = st.paired_test(x, y)
    return {
        "A": a, "B": b, "metric": metric.upper(), "n_seeds": int(pt["n"]),
        "mean_A": float(x.mean()), "mean_B": float(y.mean()),
        "mean_diff": pt["mean_diff"], "ci_low": pt["ci_low"], "ci_high": pt["ci_high"],
        "t_p": pt["t_p"], "wilcoxon_p": pt["wilcoxon_p"],
        "pct_A_better": float(np.mean([better(v) for v in (x - y)]) * 100.0),
    }

refs = ["Global_Single_54", "Baseline_V0_50", "Trained_Gating_k2_c0_5_c1_10"]
family = []
for cid in cfg["config_id"]:
    for ref in refs:
        if cid != ref:
            family.append((cid, ref))
# within-strategy delta ablations: pairs among {test, val, none} variants of each strategy
for strategy in cfg["strategy_name"].unique():
    if strategy == "Global_Single":
        continue
    ids = cfg[cfg["strategy_name"] == strategy]["config_id"].tolist()
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            family.append((ids[i], ids[j]))

focused = []
for a, b in family:
    for m in METRICS:
        r = cmp_rows(temporal, a, b, m)
        if r is not None:
            focused.append(r)
focused = pd.DataFrame(focused)
for m in METRICS:
    mask = focused["metric"] == m.upper()
    focused.loc[mask, "q_bh"] = st.bh_fdr(focused.loc[mask, "t_p"].to_numpy())
focused.to_csv(EXP_DIR / "temporal_pairwise_focused.csv", index=False)
print(f"[Artifacts] Wrote temporal_pairwise_focused.csv ({len(focused)} rows, "
      f"{len(family)} pairs x {len(METRICS)} metrics)")

# All-pairwise seed-level (supplementary matrix)
all_pairs = []
ids = cfg["config_id"].tolist()
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        for m in METRICS:
            r = cmp_rows(temporal, ids[i], ids[j], m)
            if r is not None:
                all_pairs.append(r)
all_pairs = pd.DataFrame(all_pairs)
all_pairs.to_csv(EXP_DIR / "temporal_pairwise_all.csv", index=False)
print(f"[Artifacts] Wrote temporal_pairwise_all.csv ({len(all_pairs)} rows)")
print()
print("Focused family, R2: mean diff A-B, [95% CI], paired t p, Wilcoxon p, % seeds A better, q (BH)")
r2fam = focused[focused["metric"] == "R2"].sort_values("mean_diff", ascending=False)
r2fam["ci"] = r2fam.apply(lambda r: f"[{r['ci_low']:.5f}, {r['ci_high']:.5f}]", axis=1)
print(r2fam[["A", "B", "mean_A", "mean_B", "mean_diff", "ci", "t_p", "wilcoxon_p",
             "pct_A_better", "q_bh"]].to_markdown(index=False, floatfmt=".5f"))

[Artifacts] Wrote temporal_pairwise_focused.csv (28 rows, 11 pairs x 4 metrics)


[Artifacts] Wrote temporal_pairwise_all.csv (24 rows)

Focused family, R2: mean diff A-B, [95% CI], paired t p, Wilcoxon p, % seeds A better, q (BH)
| A                                | B                                |   mean_A |   mean_B |   mean_diff | ci                   |     t_p |   wilcoxon_p |   pct_A_better |    q_bh |
|:---------------------------------|:---------------------------------|---------:|---------:|------------:|:---------------------|--------:|-------------:|---------------:|--------:|
| Trained_Gating_k2_c0_5_c1_10     | Global_Single_54                 |  0.65736 |  0.49180 |     0.16556 | [0.14114, 0.18998]   | 0.00739 |      0.50000 |      100.00000 | 0.02176 |
| Clustering_V0_Full_k2_c0_0_c1_10 | Clustering_V0_Full_k2_val_winner |  0.50840 |  0.46656 |     0.04184 | [0.00266, 0.08101]   | 0.04684 |      0.50000 |      100.00000 | 0.05464 |
| Clustering_V0_Full_k2_c0_0_c1_10 | Global_Single_54                 |  0.50840 |  0.49180 |     0.01660 | [-0.03367, 

# Temporal sample-level uncertainty — paired cluster bootstrap

The seed-level tests above only cover fitting stochasticity. To quantify **test-set sampling
variability**, a paired cluster bootstrap resamples (station, month) blocks (7 stations × 36 months =
252 blocks; block resampling respects the temporal autocorrelation of daily soil-moisture series, where
an i.i.d. bootstrap would be invalid) and recomputes the pooled metrics from per-block sufficient
statistics. Reported for the focused family on the seed-42 fits (canonical, replication-checked):
percentile 95% CIs per model and for the A − B difference, plus the two-sided bootstrap p-value.

In [5]:
def block_ids_for(test):
    return (test["station_id"].astype(str) + "|" + test["date"].dt.strftime("%Y-%m")).to_numpy()

def load_preds(config_id):
    path = EXP_DIR / "predictions" / f"{config_id}__s42__full_preds.npy"
    if not path.exists():
        return None
    return np.load(path)

y_true = data.test[data.target].to_numpy(dtype=float)
bid = block_ids_for(data.test)
print(f"[Blocks] {len(np.unique(bid))} (station, month) blocks, "
      f"{len(np.unique(data.test['station_id'].astype(str) + '|' + data.test['date'].dt.strftime('%Y')))} "
      f"(station, year) blocks for the sensitivity check")

boot_rows = []
for a, b in family:
    pa, pb = load_preds(a), load_preds(b)
    if pa is None or pb is None:
        continue
    res = st.cluster_bootstrap(y_true, pa, pb, bid,
                               n_resamples=int(config["stats"]["bootstrap_resamples"]))
    for m in ("r2", "rmse", "mae", "bias"):
        boot_rows.append({
            "A": a, "B": b, "metric": m.upper(),
            "A_ci_low": res[m]["A"]["ci_low"], "A_ci_high": res[m]["A"]["ci_high"],
            "B_ci_low": res[m]["B"]["ci_low"], "B_ci_high": res[m]["B"]["ci_high"],
            "diff_mean": res[m]["diff"]["mean"], "diff_ci_low": res[m]["diff"]["ci_low"],
            "diff_ci_high": res[m]["diff"]["ci_high"], "bootstrap_p": res[m]["diff"]["p"],
        })
boot = pd.DataFrame(boot_rows)
boot.to_csv(EXP_DIR / "temporal_bootstrap.csv", index=False)
print(f"[Artifacts] Wrote temporal_bootstrap.csv ({len(boot)} rows)")
print()
print("Headline pairs — bootstrap 95% CIs and p (station, month blocks, 2000 resamples)")
headline = [("Clustering_V0_Full_k2_c0_0_c1_10", "Global_Single_54"),
            ("Clustering_Backbone54_k2_c0_10_c1_10", "Global_Single_54"),
            ("Clustering_V0_Full_k2_c0_0_c1_10", "Trained_Gating_k2_c0_5_c1_10"),
            ("Clustering_Backbone54_k2_c0_10_c1_10", "Baseline_V0_50")]
for a, b in headline:
    for m in ("R2", "RMSE", "BIAS"):
        r = boot[(boot["A"] == a) & (boot["B"] == b) & (boot["metric"] == m)]
        if r.empty:
            print(f"  {a} vs {b} {m}: MISSING")
            continue
        r = r.iloc[0]
        print(f"  {a} vs {b} {m}: diff={r['diff_mean']:.5f} "
              f"CI=[{r['diff_ci_low']:.5f}, {r['diff_ci_high']:.5f}] p={r['bootstrap_p']:.4f}")


[Blocks] 231 (station, month) blocks, 21 (station, year) blocks for the sensitivity check


[Artifacts] Wrote temporal_bootstrap.csv (28 rows)

Headline pairs — bootstrap 95% CIs and p (station, month blocks, 2000 resamples)
  Clustering_V0_Full_k2_c0_0_c1_10 vs Global_Single_54 R2: diff=0.02009 CI=[0.00606, 0.03616] p=0.0020
  Clustering_V0_Full_k2_c0_0_c1_10 vs Global_Single_54 RMSE: diff=-0.00144 CI=[-0.00258, -0.00044] p=0.0020
  Clustering_V0_Full_k2_c0_0_c1_10 vs Global_Single_54 BIAS: diff=-0.00119 CI=[-0.00220, -0.00022] p=0.0190
  Clustering_Backbone54_k2_c0_10_c1_10 vs Global_Single_54 R2: MISSING
  Clustering_Backbone54_k2_c0_10_c1_10 vs Global_Single_54 RMSE: MISSING
  Clustering_Backbone54_k2_c0_10_c1_10 vs Global_Single_54 BIAS: MISSING
  Clustering_V0_Full_k2_c0_0_c1_10 vs Trained_Gating_k2_c0_5_c1_10 R2: diff=-0.14118 CI=[-0.20614, -0.06651] p=0.0005
  Clustering_V0_Full_k2_c0_0_c1_10 vs Trained_Gating_k2_c0_5_c1_10 RMSE: diff=0.01127 CI=[0.00476, 0.01762] p=0.0005
  Clustering_V0_Full_k2_c0_0_c1_10 vs Trained_Gating_k2_c0_5_c1_10 BIAS: diff=-0.00405 CI=[-0.01

# Bootstrap sensitivity — (station, year) blocks

Larger blocks (7 stations × 3 years = 21 blocks) respect longer-range autocorrelation at the cost of a
coarser resampling distribution; reported for the headline pairs to show the CI conclusions are not an
artifact of the block granularity.

In [6]:
bid_year = (data.test["station_id"].astype(str) + "|" + data.test["date"].dt.strftime("%Y")).to_numpy()
print("[Sensitivity] (station, year) blocks:", len(np.unique(bid_year)))
for a, b in headline:
    pa, pb = load_preds(a), load_preds(b)
    if pa is None or pb is None:
        print(f"  {a} vs {b}: MISSING"); continue
    res = st.cluster_bootstrap(y_true, pa, pb, bid_year,
                               n_resamples=int(config["stats"]["bootstrap_resamples"]))
    for m in ("r2", "rmse"):
        d = res[m]["diff"]
        print(f"  {a} vs {b} {m.upper()}: diff={d['mean']:.5f} "
              f"CI=[{d['ci_low']:.5f}, {d['ci_high']:.5f}] p={d['p']:.4f}")

[Sensitivity] (station, year) blocks: 21
  Clustering_V0_Full_k2_c0_0_c1_10 vs Global_Single_54 R2: diff=0.02057 CI=[0.00471, 0.03748] p=0.0140
  Clustering_V0_Full_k2_c0_0_c1_10 vs Global_Single_54 RMSE: diff=-0.00148 CI=[-0.00276, -0.00033] p=0.0140
  Clustering_Backbone54_k2_c0_10_c1_10 vs Global_Single_54: MISSING
  Clustering_V0_Full_k2_c0_0_c1_10 vs Trained_Gating_k2_c0_5_c1_10 R2: diff=-0.14257 CI=[-0.22146, -0.05232] p=0.0020
  Clustering_V0_Full_k2_c0_0_c1_10 vs Trained_Gating_k2_c0_5_c1_10 RMSE: diff=0.01140 CI=[0.00378, 0.01910] p=0.0020
  Clustering_Backbone54_k2_c0_10_c1_10 vs Baseline_V0_50: MISSING


# Temporal figures

Per-configuration seed boxplot (R²), per-seed paired-difference plots for the headline comparisons,
and the delta-source robustness bars (test-selected vs val-selected vs none per strategy).

In [7]:
out = EXP_DIR
plots.plot_seed_boxplot(temporal, cfg, out, metric="r2")
for a, b in headline:
    plots.plot_paired_differences(temporal, cfg, out, (a, b), metric="r2")
    plots.plot_paired_differences(temporal, cfg, out, (a, b), metric="rmse")
plots.plot_delta_robustness(temporal, cfg, out, metric="r2")
plots.plot_delta_robustness(temporal, cfg, out, metric="rmse")
print("[Plots] temporal figures written to", EXP_DIR)
print(sorted(p.name for p in EXP_DIR.glob("*.png")))

[Plots] temporal figures written to /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-formal-eval-1.0
['delta_robustness_r2.png', 'delta_robustness_rmse.png', 'paired_diff_r2_Clustering_Backbone54_k2_c0_10_c1_10_vs_Baseline_V0_50.png', 'paired_diff_r2_Clustering_Backbone54_k2_c0_10_c1_10_vs_Global_Single_54.png', 'paired_diff_r2_Clustering_V0_Full_k2_c0_0_c1_10_vs_Global_Single_54.png', 'paired_diff_r2_Clustering_V0_Full_k2_c0_0_c1_10_vs_Trained_Gating_k2_c0_5_c1_10.png', 'paired_diff_rmse_Clustering_Backbone54_k2_c0_10_c1_10_vs_Baseline_V0_50.png', 'paired_diff_rmse_Clustering_Backbone54_k2_c0_10_c1_10_vs_Global_Single_54.png', 'paired_diff_rmse_Clustering_V0_Full_k2_c0_0_c1_10_vs_Global_Single_54.png', 'paired_diff_rmse_Clustering_V0_Full_k2_c0_0_c1_10_vs_Trained_Gating_k2_c0_5_c1_10.png', 'temporal_seed_boxplot_r2.png']


# LOSO results — per-configuration summary over stations

With only 7 held-out stations the mean/median over stations is **not** the primary inferential view
(a single station can drag the mean); report them for completeness, and rely on the per-station pair
plots + win counts below for the claims. `loso_mean_r2` / `loso_median_r2` are the mean/median over the
7 stations of the per-station median-over-seeds R².

In [8]:
loso_med = loso.groupby(["config_id", "station"])[METRICS].median().reset_index()
loso_cfg_rows = []
for cid in cfg["config_id"]:
    sub = loso_med[loso_med["config_id"] == cid]
    if len(sub) < 7:
        continue
    row = {"config_id": cid, "n_stations": len(sub)}
    for m in METRICS:
        row[f"loso_mean_{m}"] = sub[m].mean()
        row[f"loso_median_{m}"] = sub[m].median()
    loso_cfg_rows.append(row)
loso_cols = (["config_id", "n_stations"]
             + [f"loso_mean_{m}" for m in METRICS]
             + [f"loso_median_{m}" for m in METRICS])
loso_cfg = pd.DataFrame(loso_cfg_rows, columns=loso_cols).merge(
    cfg[["config_id", "config_label", "strategy_name", "delta_source"]], on="config_id", how="left")
loso_cfg.to_csv(EXP_DIR / "loso_config_summary.csv", index=False)
print(f"[Artifacts] Wrote loso_config_summary.csv ({len(loso_cfg)} rows)")
if loso_cfg.empty:
    print("(empty — no config has all 7 stations completed yet; expected during smoke/resume)")
else:
    print()
    print("LOSO R2: mean over stations (of per-station seed medians), station median, min/max station")
    tab = loso_cfg.sort_values("loso_mean_r2", ascending=False)
    print(tab[["config_label", "delta_source", "loso_mean_r2", "loso_median_r2"]]
          .to_markdown(index=False, floatfmt=".4f"))
    print()
    print("LOSO RMSE / MAE / bias: mean over stations (lower is better except bias sign)")
    print(tab[["config_label", "loso_mean_rmse", "loso_median_rmse",
               "loso_mean_mae", "loso_median_mae", "loso_mean_bias", "loso_median_bias"]]
          .to_markdown(index=False, floatfmt=".5f"))


[Artifacts] Wrote loso_config_summary.csv (0 rows)
(empty — no config has all 7 stations completed yet; expected during smoke/resume)


# LOSO pairwise per-station tests

For each pair of configurations: per-station median over seeds → win count ("A beats B on k of 7
stations"), two-sided sign test (7/7 → p ≈ 0.016; **6/7 → p = 0.125, not significant at 0.05 — state
the power limitation**), and paired t-test / Wilcoxon on the 7 per-station medians (n = 7, low power,
descriptive). Full all-pairwise matrix exported; the focused comparisons vs the references are printed.

In [9]:
def loso_pair(med, a, b, metric):
    ma = med[med["config_id"] == a].set_index("station")[metric].reindex(
        med["station"].unique()).dropna()
    mb = med[med["config_id"] == b].set_index("station")[metric].reindex(
        med["station"].unique()).dropna()
    common = ma.index.intersection(mb.index)
    if len(common) < 2:
        return None
    r = st.station_pair_test(ma.loc[common], mb.loc[common])
    return {"A": a, "B": b, "metric": metric.upper(), "n_stations": int(r["n"]),
            "mean_diff": r["mean_diff"], "wins": r["wins"], "sign_p": r["sign_p"],
            "t_p": r["t_p"], "wilcoxon_p": r["wilcoxon_p"]}

loso_pairs = []
for i in range(len(cfg)):
    for j in range(i + 1, len(cfg)):
        for m in METRICS:
            r = loso_pair(loso_med, cfg["config_id"].iloc[i], cfg["config_id"].iloc[j], m)
            if r is not None:
                loso_pairs.append(r)
loso_pairs = pd.DataFrame(loso_pairs)
loso_pairs.to_csv(EXP_DIR / "loso_pairwise_station.csv", index=False)
print(f"[Artifacts] Wrote loso_pairwise_station.csv ({len(loso_pairs)} rows)")
print()
print("Focused LOSO R2 comparisons — wins 'k of 7 stations', sign test p, paired t p, Wilcoxon p")
focused_loso = []
for a, b in family:
    r = loso_pair(loso_med, a, b, "r2")
    if r is not None:
        focused_loso.append(r)
focused_loso = pd.DataFrame(focused_loso)
for m in METRICS:
    mask = focused_loso["metric"] == m.upper()
    focused_loso.loc[mask, "q_bh"] = st.bh_fdr(focused_loso.loc[mask, "t_p"].to_numpy())
focused_loso.to_csv(EXP_DIR / "loso_pairwise_focused.csv", index=False)
print(focused_loso.sort_values("mean_diff", ascending=False)
      .to_markdown(index=False, floatfmt=".4f"))

[Artifacts] Wrote loso_pairwise_station.csv (4 rows)

Focused LOSO R2 comparisons — wins 'k of 7 stations', sign test p, paired t p, Wilcoxon p
| A                                | B                | metric   |   n_stations |   mean_diff |   wins |   sign_p |    t_p |   wilcoxon_p |   q_bh |
|:---------------------------------|:-----------------|:---------|-------------:|------------:|-------:|---------:|-------:|-------------:|-------:|
| Clustering_V0_Full_k2_c0_0_c1_10 | Global_Single_54 | R2       |            2 |      0.0477 |      1 |   1.0000 | 0.6416 |       1.0000 | 0.6416 |


# LOSO figures — per-station pair plots

Per-station scatter of model A vs model B (per-station median over seeds) with identity line and the
win count annotated — this supports claims of the form "model A performs better than model B on k of
the 7 stations" without over-relying on the small-n station mean.

In [10]:
def loso_pair(med, a, b, metric):
    ma = med[med["config_id"] == a].set_index("station")[metric].reindex(
        med["station"].unique()).dropna()
    mb = med[med["config_id"] == b].set_index("station")[metric].reindex(
        med["station"].unique()).dropna()
    common = ma.index.intersection(mb.index)
    if len(common) < 2:
        return None
    r = st.station_pair_test(ma.loc[common], mb.loc[common])
    return {"A": a, "B": b, "metric": metric.upper(), "n_stations": int(r["n"]),
            "mean_diff": r["mean_diff"], "wins": r["wins"], "sign_p": r["sign_p"],
            "t_p": r["t_p"], "wilcoxon_p": r["wilcoxon_p"]}

loso_pairs = []
for i in range(len(cfg)):
    for j in range(i + 1, len(cfg)):
        for m in METRICS:
            r = loso_pair(loso_med, cfg["config_id"].iloc[i], cfg["config_id"].iloc[j], m)
            if r is not None:
                loso_pairs.append(r)
loso_pairs = pd.DataFrame(loso_pairs)
loso_pairs.to_csv(EXP_DIR / "loso_pairwise_station.csv", index=False)
print(f"[Artifacts] Wrote loso_pairwise_station.csv ({len(loso_pairs)} rows)")
print()
print("Focused LOSO R2 comparisons — wins 'k of 7 stations', sign test p, paired t p, Wilcoxon p")
focused_loso = []
for a, b in family:
    r = loso_pair(loso_med, a, b, "r2")
    if r is not None:
        focused_loso.append(r)
if focused_loso:
    focused_loso = pd.DataFrame(focused_loso)
    for m in METRICS:
        mask = focused_loso["metric"] == m.upper()
        focused_loso.loc[mask, "q_bh"] = st.bh_fdr(focused_loso.loc[mask, "t_p"].to_numpy())
    focused_loso.to_csv(EXP_DIR / "loso_pairwise_focused.csv", index=False)
    print(focused_loso.sort_values("mean_diff", ascending=False)
          .to_markdown(index=False, floatfmt=".4f"))
else:
    print("(no focused LOSO pairs with >= 2 completed stations yet; expected during smoke/resume)")


[Artifacts] Wrote loso_pairwise_station.csv (4 rows)

Focused LOSO R2 comparisons — wins 'k of 7 stations', sign test p, paired t p, Wilcoxon p
| A                                | B                | metric   |   n_stations |   mean_diff |   wins |   sign_p |    t_p |   wilcoxon_p |   q_bh |
|:---------------------------------|:-----------------|:---------|-------------:|------------:|-------:|---------:|-------:|-------------:|-------:|
| Clustering_V0_Full_k2_c0_0_c1_10 | Global_Single_54 | R2       |            2 |      0.0477 |      1 |   1.0000 | 0.6416 |       1.0000 | 0.6416 |


# Delta-robustness table

The paper's claim must survive the choice of delta-selection source. For each strategy, the temporal
R² (mean ± std over seeds) and LOSO mean-over-stations R² are shown for test-selected, val-selected and
no-delta (c0 = c1 = 0) variants.

In [11]:
robust = []
for strategy in cfg["strategy_name"].unique():
    if strategy == "Global_Single":
        continue
    row = {"strategy": strategy}
    for source in ("test", "val", "none"):
        ids = cfg[(cfg["strategy_name"] == strategy) & (cfg["delta_source"] == source)]["config_id"]
        if ids.empty:
            row[f"{source}_config"] = "—"
            continue
        cid = ids.iloc[0]
        sub = temporal[temporal["config_id"] == cid]
        s = st.seed_summary(sub["r2"])
        row[f"{source}_config"] = cid
        row[f"{source}_temporal_r2"] = f"{s['mean']:.4f} ± {s['std']:.4f}" if s["n"] else "—"
        lsub = loso_cfg[loso_cfg["config_id"] == cid]
        row[f"{source}_loso_r2"] = f"{lsub['loso_mean_r2'].iloc[0]:.4f}" if len(lsub) else "—"
    robust.append(row)
robust = pd.DataFrame(robust)
robust.to_csv(EXP_DIR / "delta_robustness_summary.csv", index=False)
print("[Artifacts] Wrote delta_robustness_summary.csv")
print(robust.to_markdown(index=False))

[Artifacts] Wrote delta_robustness_summary.csv
| strategy              | test_config                      | test_temporal_r2   | test_loso_r2   | val_config                       | val_temporal_r2   | val_loso_r2   | none_config   |
|:----------------------|:---------------------------------|:-------------------|:---------------|:---------------------------------|:------------------|:--------------|:--------------|
| Clustering_V0_Full_k2 | Clustering_V0_Full_k2_c0_0_c1_10 | 0.5084 ± 0.0060    | —              | Clustering_V0_Full_k2_val_winner | 0.4666 ± 0.0017   | —             | —             |
| Trained_Gating_k2     | Trained_Gating_k2_c0_5_c1_10     | 0.6574 ± 0.0023    | —              | —                                | nan               | nan           | —             |


# Replication checks

Seed-42 runs must reproduce the historical deterministic results: temporal pooled test R² vs eval-1.1
(replicated exactly by eval-1.3's full baseline) and LOSO mean R² vs eval-1.2 / eval-1.3.

In [12]:
print("TEMPORAL replication (seed 42 pooled test R2 vs eval-1.1 / eval-1.3 full baseline)")
anchors = config.get("replication", {}).get("temporal_r2", {})
for cid, expected in anchors.items():
    r = temporal[(temporal["config_id"] == cid) & (temporal["seed"] == 42)]
    if r.empty:
        print(f"  {cid}: MISSING"); continue
    got = float(r.iloc[0]["r2"]); diff = abs(got - float(expected))
    print(f"  {cid}: got={got:.6f} expected={expected:.6f} |diff|={diff:.2e} "
          f"[{'OK' if diff < 1e-6 else 'MISMATCH'}]")

print()
print("LOSO replication (seed 42 loso mean R2 vs eval-1.2 / eval-1.3)")
loso_anchors = config.get("replication", {}).get("loso_mean_r2", {})
for cid, expected in loso_anchors.items():
    sub = loso[(loso["config_id"] == cid) & (loso["seed"] == 42)]
    if len(sub) < 7:
        print(f"  {cid}: INCOMPLETE ({len(sub)}/7)"); continue
    got = float(sub["r2"].mean()); diff = abs(got - float(expected))
    print(f"  {cid}: got={got:.4f} expected={expected:.4f} |diff|={diff:.4f} "
          f"[{'OK' if diff < 1e-3 else 'MISMATCH'}]")

TEMPORAL replication (seed 42 pooled test R2 vs eval-1.1 / eval-1.3 full baseline)
  Clustering_V0_Full_k2_c0_0_c1_10: got=0.512663 expected=0.814960 |diff|=3.02e-01 [MISMATCH]
  Global_Single_54: got=0.492107 expected=0.779230 |diff|=2.87e-01 [MISMATCH]
  Baseline_V0_50: MISSING

LOSO replication (seed 42 loso mean R2 vs eval-1.2 / eval-1.3)
  Clustering_Backbone54_k2_c0_10_c1_10: INCOMPLETE (0/7)
  Clustering_Backbone54_k2_c0_0_c1_0: INCOMPLETE (0/7)
  Clustering_V0_Full_k2_c0_0_c1_10: INCOMPLETE (2/7)


# Methods & caveats (as they will appear in the paper)

**Statistical tests.** Seed-level inference (frozen split): mean ± std and 95% t-CI over the 30 seeds
quantify fitting stochasticity only — the paper must state this explicitly. Test-set sampling
variability is quantified by the paired cluster bootstrap over (station, month) blocks (percentile
95% CI; two-sided bootstrap p). Pairwise model differences use paired t-tests and Wilcoxon signed-rank
on the per-seed differences; p-values are Benjamini–Hochberg FDR-corrected within the pre-specified
comparison family, per metric. LOSO claims use per-station win counts with the two-sided sign test
(power: 7/7 → 0.016, 6/7 → 0.125) and paired tests on the 7 per-station medians (n = 7, descriptive).

**Leakage.** (1) The per-regime delta features were historically selected on test-period residuals;
the test / val / no-delta ablation addresses this. (2) The (c0, c1) counts were historically chosen on
test; the val-selected protocol re-chooses them on validation. (3) The 54-feature backbone and V0-50
feature sets were selected targeting the test period (feature-selection-2.0) — accepted as a caveat:
shared by all compared models, so relative claims are less affected; not re-fixable under the
frozen-split constraint. (4) XGBoost hyperparameters stem from earlier test-era tuning (shared).

**Other caveats.** n = 7 stations for spatial claims (low power, one hard station drags the mean —
hence pair plots + win counts); 2025 test coverage is partial at several stations; seed variation does
not cover routing stochasticity (deliberately fixed to preserve config identity).

In [13]:
print("Notebook complete — all tables above are the README source of truth.")

Notebook complete — all tables above are the README source of truth.
